# MS4215 · Spotify Data Detective
## A real investigation in Python — not a worksheet of commands

**The brief:** A music discovery team asks you: *What makes a track energetic, and how confident can we be in what these data tell us?*

Your job is to deliver an evidence-based explanation, not just code.

**The investigative cycle:** Ask → Inspect → Question → Explore → Model → Challenge → Communicate.

**What makes this different from the original university Lab 1?** We use the same Spotify file, but devise our own realistic research questions, analyse its sampling and duplicate-record problems, compare competing models on unseen music, and learn how an apparently persuasive result can mislead.

### Your learning contract
1. **PREDICT** before running a cell.
2. **OBSERVE** the output, including anything surprising.
3. **EXPLAIN** it without technical jargon first.
4. **CHALLENGE** the conclusion before accepting it.

**Start with the four investigation questions:** Is a loud song necessarily energetic? Does energy predict popularity? Do genres differ? Can we predict an unseen track's energy score?

**Important:** Spotify's "energy" is an existing computed audio-feature score on a scale from 0 to 1. This investigation examines *associations among provider-defined features*; it does **not** establish what makes humans feel energised or what causes musical popularity.

**Level:** Start from fundamentals, finish with a credible portfolio case study. Run cells in order in Colab or Jupyter using Shift+Enter.


# ACT 1 — Get a feel for the dataset

Before seeing any formula: **which of these pairs do you expect to be most strongly related?**

- Energy and loudness
- Energy and acousticness
- Energy and popularity

Write your order down. We will test it later.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Only one download. Local copies take priority for offline work.
locations = [
    Path("data/lab1_dataset_spotify.csv"),
    Path("../data/lab1_dataset_spotify.csv"),
    Path("lab1_dataset_spotify.csv")
]
local_file = next((p for p in locations if p.is_file()), None)
if local_file:
    raw = pd.read_csv(local_file)
    print("Loaded local dataset:", local_file)
else:
    source = ("https://raw.githubusercontent.com/ErikDownes/"
              "ErikDownes.github.io/main/resources/ms4215/archives/"
              "lab1_dataset_spotify.zip")
    with ZipFile(BytesIO(urlopen(source).read())) as z:
        csv_members = [p for p in z.namelist() if p.lower().endswith(".csv")]
        if not csv_members:
            raise FileNotFoundError("No CSV was found in the official course archive.")
        with z.open(csv_members[0]) as csv:
            raw = pd.read_csv(csv)
    print("Loaded original course dataset from Erik's website")

print("Dataset size:", f"{len(raw):,}", "rows")
display(raw.head(5))


## What's a row? What's a variable?

A **row** is a recording of a track in a genre category. Crucially, that may *not* mean one unique song.

Examples of **numeric variables**: energy (0–1), tempo (beats per minute), loudness (decibels), and popularity (0–100).

Examples of **categories**: genre (nominal), explicit-content flag (binary).

**Before you run:** What should a dataset with 114,000 rows and 114 genres look like if it represents a random sample of the world's music? What if each genre has exactly the same count?


In [ ]:
print("Rows:", f"{len(raw):,}")
print("Columns:", raw.shape[1])
print("Distinct track IDs:", f"{raw['track_id'].nunique():,}")
print("Number of genre labels:", raw['track_genre'].nunique())

genre_counts = raw["track_genre"].value_counts()
print("\nHow many records does each genre have?")
display(genre_counts.describe().to_frame("number of rows"))

print("\nRepeated rows if we expect one row per track:")
print("Repeat track IDs:", f"{raw.duplicated('track_id').sum():,}")
print("Exact duplicate rows:", f"{raw.duplicated().sum():,}")

print("\nHow many track IDs are listed under more than one genre?")
print(f"{(raw.groupby('track_id')['track_genre'].nunique() > 1).sum():,}")


### Stop and think: a critical discovery

**This file has exactly 1,000 rows for each of 114 genres.** That's not representative of the proportion of genres listeners actually play, and tracks can appear more than once.

This is the point where a competent analyst changes the design of the analysis.

- For a question about **genre-labelled entries**, retain the genre-labelled rows and describe them as such.
- For a question about **unique tracks**, keep one record per track ID.
- For a **train/test comparison**, do not allow the same track ID to appear in both sets.
- Do not claim this dataset is a random sample of the entire Spotify catalogue.

**Write down:** Why could a model appear to be more accurate if a repeated track appears in its training *and* test data?


# ACT 2 — Is the data trustworthy?

Don't clean just because a function can clean. Inspect first, decide what each problem means, and keep a record.

**Three kinds of "nothing":**
- Zero energy is a recorded value, not missing.
- A missing track name is unknown text, not a zero.
- Zero duration may be a problematic record; *investigate*, don't invent its duration.


In [ ]:
missing = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_percent": (100 * raw.isna().mean()).round(2)
})
display(missing)

quantities = ["energy", "loudness", "acousticness", "danceability",
              "valence", "tempo", "duration_ms", "popularity"]
display(raw[quantities].agg(["min", "median", "max"]).T)
print("Tracks with zero duration:", int((raw["duration_ms"] == 0).sum()))
print("Tracks with zero tempo:", int((raw["tempo"] == 0).sum()))


## The unit-of-analysis decision

For song-level modelling, we'll make a table with **one row per track ID**. We are not saying duplicate genres were "mistakes"—we are saying a repeated song is not an independent new song.

Later, for *genre-specific analysis*, we'll go back to the original entries, because removing genre labels would answer a different question.

**Predict:** Should the average energy change when a popular multi-genre track is counted once instead of several times?


In [ ]:
tracks = raw.drop_duplicates("track_id").copy()
tracks["duration_minutes"] = tracks["duration_ms"] / 60_000

comparison = pd.DataFrame({
    "genre-labelled rows": [len(raw), raw["energy"].mean()],
    "unique tracks": [len(tracks), tracks["energy"].mean()]
}, index=["observations", "mean energy"])
display(comparison.round(3))
assert tracks["track_id"].is_unique
print("Clean song-level table:", tracks.shape)


### Mini-experiment — challenge a cleaning decision

Try **not** deleting zero-tempo tracks immediately. First look at a few; some may be spoken-word or very quiet recordings. Compare their energy and genre distributions with other tracks.

**Question:** What evidence would convince you to remove a record, rather than leave it and report a limitation?


In [ ]:
unusual = tracks.loc[(tracks["tempo"] == 0) | (tracks["duration_ms"] == 0)]
display(unusual[["track_name", "track_genre", "tempo",
                 "duration_ms", "energy"]].head(12))
print("Potentially unusual records:", len(unusual))


# ACT 3 — Explore the story before fitting a model

Three simple graphs beat one intimidating table. We will learn **distribution**, **centre**, **spread** and **outliers**.

**Predict:** Will popularity be tightly clustered around one value, or spread across the 0–100 scale? Will energy look symmetrical?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feature in zip(axes, ["energy", "loudness", "popularity"]):
    ax.hist(tracks[feature].dropna(), bins=35, edgecolor="white")
    ax.set(title=f"Distribution of {feature}",
           xlabel=feature, ylabel="Unique tracks")
fig.suptitle("What kinds of values are in our dataset?", fontsize=13)
plt.tight_layout()
plt.show()

display(tracks[["energy", "loudness", "popularity"]].describe().T.round(3))


## A mathematical decision — mean or median?

- **Mean:** balances all values, but responds strongly to extreme observations.
- **Median:** divides sorted observations in half, often a better notion of "typical" with skew.
- **IQR:** 75th percentile minus 25th percentile, describing the middle half of observations.

**Question:** If one track became enormously popular, which summary would move more: mean popularity or median popularity? Why?


In [ ]:
def summaries(frame, columns):
    out = frame[columns].agg(["mean", "median", "std"]).T
    out["IQR"] = frame[columns].quantile(.75) - frame[columns].quantile(.25)
    return out.round(3)

display(summaries(tracks, ["energy", "loudness", "popularity", "tempo"]))


## Experiment: is the same idea called "energetic" in every genre?

We compare **genre-labelled rows**, not a probability sample of all music listeners. Genres with 1,000 records each are *equal-sized by design*, not equal-sized in the world.

**Predict:** Which would have a higher median energy score: classical, acoustic, pop, dance, hip-hop, jazz, ambient or heavy-metal?

**Important:** A difference between genres does not show that changing a track's genre label would change the audio energy score.


In [ ]:
selected_genres = ["classical", "acoustic", "pop", "dance",
                   "hip-hop", "jazz", "ambient", "heavy-metal"]
genre_entries = raw.loc[raw["track_genre"].isin(selected_genres)].copy()
genre_energy = (genre_entries.groupby("track_genre")["energy"]
                .agg(count="count", median="median", mean="mean")
                .sort_values("median"))
display(genre_energy.round(3))

ax = genre_energy["median"].plot.barh(figsize=(9, 5))
ax.set(title="Median audio energy in selected genre-labelled entries",
       xlabel="Median energy (0–1)", ylabel="")
plt.tight_layout()
plt.show()


## Association is not causation

**Claim:** "Louder songs have higher energy scores."

What does **have higher** mean? A relationship in observed data does not mean turning up the volume would *cause* the provider's energy score to rise.

Likewise, if popularity and energy are unrelated in this dataset, that does not tell us whether an advertising campaign affects popularity.

**Predict before seeing numbers:** rank loudness, acousticness, tempo, danceability, valence and popularity by their likely absolute correlation with energy.


In [ ]:
candidates = ["loudness", "acousticness", "tempo", "danceability",
              "valence", "popularity", "energy"]
r = tracks[candidates].corr()["energy"].drop("energy")
result = pd.DataFrame({"Pearson r": r, "strength |r|": r.abs()})
display(result.sort_values("strength |r|", ascending=False).round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, x in zip(axes, ["loudness", "acousticness", "popularity"]):
    subset = tracks[[x, "energy"]].sample(n=2500, random_state=12)
    ax.scatter(subset[x], subset["energy"], s=8, alpha=.18)
    ax.set(xlabel=x, ylabel="energy", title=f"Energy versus {x}")
plt.tight_layout()
plt.show()


### Don't be fooled by large numbers

A correlation is a **measure of linear association**, not an accuracy score and not a causal estimate.

**Reflect:** If popularity has little linear association with energy, is it sensible to suggest that energetic music will automatically get more streams? No. What other factors might determine popularity?

**Extra investigation:** Are correlations the same within classical, pop and heavy-metal? We can check rather than assume.


In [ ]:
for genre in ["classical", "pop", "heavy-metal"]:
    sub = raw.loc[raw["track_genre"].eq(genre)]
    print(f"{genre:12} | loudness–energy: {sub['loudness'].corr(sub['energy']):+.3f}"
          f" | popularity–energy: {sub['popularity'].corr(sub['energy']):+.3f}")


# ACT 4 — Make a prediction and put it at risk

Imagine the energy score is **unavailable for some new tracks**. Could we estimate it from other measured characteristics?

**Important:** In the actual dataset, energy is already supplied. We are building a **learning simulation**, not claiming a real platform needs to reconstruct its own known value.

A credible model must be tested on tracks it has never been trained on.

We will compare:
1. A **baseline**: predict the training-set mean every time.
2. A **simple model**: use loudness only.
3. A **multivariable model**: use loudness, acousticness, danceability, valence and tempo.

The question is whether added complexity **improves predictions on unseen tracks**.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score

features = ["loudness", "acousticness", "danceability", "valence", "tempo"]
frame = tracks.dropna(subset=["energy", *features]).copy()

train, test = train_test_split(frame, test_size=0.25, random_state=42)
assert set(train["track_id"]).isdisjoint(set(test["track_id"]))
print("Train tracks:", len(train), "Test tracks:", len(test))
print("Overlap in track IDs:", len(set(train["track_id"]) & set(test["track_id"])))
print("Important: test labels are NOT used to fit the models.")


## MAE: explain a model error in human language

**MAE = mean absolute error**, the average distance between the predicted energy and the actual score on the holdout tracks.

Since energy lives on a 0–1 scale, **MAE of 0.10** means the average prediction misses by about 0.10 energy units.

**R²** tells us how much better the model's squared errors are compared with predicting the **test-set mean**. R² can be negative on unseen data; it is not a percentage of individually correct predictions.

**Predict:** Which of our three models will have the lowest test MAE?


In [ ]:
baseline = np.full(len(test), train["energy"].mean())
simple = LinearRegression().fit(train[["loudness"]], train["energy"])
simple_pred = simple.predict(test[["loudness"]])

multi = Ridge(alpha=1.0).fit(train[features], train["energy"])
multi_pred = multi.predict(test[features])

predictions = {
    "Training-mean baseline": baseline,
    "Loudness only": simple_pred,
    "Five-feature Ridge model": multi_pred
}
scores = pd.DataFrame({
    "MAE (lower better)": {
        name: mean_absolute_error(test["energy"], yhat)
        for name, yhat in predictions.items()
    },
    "Test R² (higher better)": {
        name: r2_score(test["energy"], yhat)
        for name, yhat in predictions.items()
    }
})
display(scores.sort_values("MAE (lower better)").round(4))


## Inspect failures, not just average accuracy

A model with decent average accuracy may systematically miss the least energetic tracks, or predict impossible values such as energy greater than 1.

In a good professional report, show *where* the model fails.

**Predict:** Will the error size be identical for low-energy, medium-energy and high-energy songs?


In [ ]:
review = test[["track_id", "energy"]].copy()
review["prediction"] = multi_pred
review["error"] = review["energy"] - review["prediction"]
review["absolute_error"] = review["error"].abs()
review["energy_band"] = pd.cut(
    review["energy"], bins=[-0.001, 0.33, 0.66, 1.001],
    labels=["low", "medium", "high"])
display(review.groupby("energy_band", observed=True)["absolute_error"]
        .agg(["count", "mean", "median"]).round(3))
print("Predictions outside the valid [0,1] energy range:",
      int(((review["prediction"] < 0) | (review["prediction"] > 1)).sum()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(review["energy"], review["prediction"], alpha=.15, s=8)
axes[0].plot([0, 1], [0, 1], linestyle="--")
axes[0].set(title="Actual vs predicted", xlabel="Actual energy",
            ylabel="Predicted energy", xlim=(0,1))
axes[1].scatter(review["prediction"], review["error"], alpha=.15, s=8)
axes[1].axhline(0, linestyle="--")
axes[1].set(title="Residuals vs fitted", xlabel="Predicted energy",
            ylabel="Actual minus predicted")
plt.tight_layout()
plt.show()


## A real analyst's danger: data leakage

If a track is listed several times under different genres, a naïve random split of all **114,000 rows** could place that same track in both training and test data. That would make the test less independent.

We avoided this by retaining **one row per unique track before splitting**.

**Advanced extension:** Keep all genre-labelled rows and use GroupShuffleSplit(groups = track_id). Would the predictive comparison change? Do not report a leakage-free result if you haven't checked track-ID overlap.


# ACT 5 — Challenge your own findings

## What if we chose a different random sample?

A defensible result should not depend entirely on a lucky split.

Run five different train/test splits. Does the conclusion that multiple features outperform the baseline remain reasonably stable? **Look at the range, not just one attractive number.**


In [ ]:
repeat_scores = []
for seed in [3, 11, 21, 42, 99]:
    a, b = train_test_split(frame, test_size=.25, random_state=seed)
    fitted = Ridge(alpha=1.0).fit(a[features], a["energy"])
    guess = fitted.predict(b[features])
    repeat_scores.append({
        "random_seed": seed,
        "test_MAE": mean_absolute_error(b["energy"], guess),
        "test_R2": r2_score(b["energy"], guess)
    })
repeat_results = pd.DataFrame(repeat_scores)
display(repeat_results.round(4))
print("MAE range:",
      round(repeat_results["test_MAE"].min(), 4),
      "to", round(repeat_results["test_MAE"].max(), 4))


## A thought experiment is not a causal experiment

Suppose we hold acousticness, danceability, valence and tempo fixed, then change the loudness value in the model. Its predicted energy will change.

That is what the **fitted statistical model** does. It does **not** prove that physically turning up a recording changes the true track's energy.

Change the numbers below only to make **plausible comparisons within the observed ranges**. A model can predict poorly when supplied with combinations it barely saw in training.


In [ ]:
# Change ONE value, then compare with the unedited example.
song_a = {
    "loudness": -15.0,
    "acousticness": 0.65,
    "danceability": 0.55,
    "valence": 0.50,
    "tempo": 110.0
}
song_b = dict(song_a)
song_b["loudness"] = -7.0

experiments = pd.DataFrame([song_a, song_b], index=["A", "B"])
experiments["predicted_energy"] = multi.predict(experiments[features])
display(experiments.round(3))
print("This describes an association estimated by the model, not a causal effect.")


## Make a defensible recommendation

**Audience:** A music playlist curator who has no interest in Python or statistical jargon.

Complete this short report with numbers taken from *your own* outputs.

> **Question** — We investigated how measured audio features relate to Spotify's energy score.
>
> **What we found** — The strongest positive linear association with energy was ____; the strongest negative was ____. Popularity's association with energy was ____.
>
> **What we tested** — A baseline prediction had test MAE ____; a model using five features had test MAE ____.
>
> **Why we should be cautious** — The source contains repeated track IDs and an equal-sized sample for each genre. The energy score is algorithm-defined, and correlations do not establish causality.
>
> **Recommendation** — We can use these features to ____ but we should not conclude ____.

### Self-assessment (0 / 1 / 2 for each)
- I defined the **unit of observation**, not just the number of rows.
- I distinguished **zero** from **missing** and treated duplicates deliberately.
- I justified a graph and a summary statistic.
- I interpreted correlation without claiming causation.
- I tested predictions on truly **unseen unique tracks**.
- I inspected model failures and clearly stated limitations.

**12/12:** defensible analysis. **8–11:** revisit your weakest explanations. **Below 8:** redo a stage with a different prediction before using code.

The final product is not a notebook that ran successfully. It is a conclusion you can explain and defend.


# ACT 6 — Independent analyst challenges

This is where the scaffolding fades. **Choose one** and complete it without copying the previous cells unchanged.

### Mission A — Music curator
A client wants to find high-energy genres among: acoustic, classical, dance, heavy-metal, hip-hop and pop.
- Define what "high energy" means, using the score.
- Show *two* different representations of the genre comparison.
- Explain why equal-sized genre samples do **not** describe real market share.
- Deliver a four-sentence conclusion, with at least one limitation.

### Mission B — Data quality investigator
A stakeholder claims: "114,000 records means 114,000 different songs."
- Quantify precisely how misleading that claim is.
- Investigate tracks assigned to more than one genre.
- Show whether collapsing by track ID changes the mean energy.
- Explain why a genre analysis and a song-level analysis may require **different tables**.

### Mission C — Predictive-model reviewer
A developer reports a superb training accuracy and says the model is ready to launch.
- Set aside a holdout of **unseen unique tracks**.
- Compare a baseline, loudness-only model and multifeature model using **MAE**.
- Examine error separately for low- and high-energy tracks.
- Explain the danger of data leakage and of extrapolating beyond the energy scale.
- Write a decision: **approve / revise / reject**, and defend it.

### Bonus: the most important question
**Do we actually need to predict Spotify's energy score if Spotify has already supplied it?** Under what real-world circumstances would a similar predictive model be useful?

An analyst's strongest answer is sometimes *"this isn't the right problem to solve."*


In [ ]:
# YOUR INDEPENDENT ANALYSIS — start with a question, not a formula.
my_question = "Write your question here"
my_prediction = "Write what you expect to find"
my_evidence = "Which columns, charts and calculations do you need?"
my_limitations = "What would make the conclusion misleading?"

print("QUESTION:", my_question)
print("PREDICTION:", my_prediction)
print("EVIDENCE PLAN:", my_evidence)
print("POSSIBLE LIMITATIONS:", my_limitations)

# Write your own analysis below this line.


<details>
<summary><strong>Click only after attempting the activities — interpretation guide</strong></summary>

- A large number of rows does not establish a random or representative sample. Here every genre has exactly 1,000 entries.
- Repeated track IDs mean row count and unique-track count are **different quantities**.
- Energy and loudness tend to be positively associated; energy and acousticness negatively associated. Run the code for the exact coefficients.
- Popularity has almost no **overall linear** association with energy in this dataset. That does not rule out nonlinear or subgroup patterns.
- A multifeature model should be compared with a simple baseline using unseen tracks, not training accuracy alone.
- The dataset does not establish causal effects, real-world genre popularity, or the subjective feeling a listener experiences.
- Learning to decide **what not to claim** is as important as learning Python.

</details>


## Link to the original university course

This notebook is an **original Python teaching investigation** built from the authentic MS4215 Spotify dataset. It complements, but does not reproduce or substitute for, the original R assignment.

**Next step in the taught sequence:** espresso extraction and the algebra of multiple regression. The mathematical machinery will make more sense now that you have used regression to solve and criticise a real question.
